# Extract RGB-IR Data from CaBuAr

Extract 4-channel RGB-IR (B02, B03, B04, B08) from Sentinel-2 CaBuAr dataset for transfer learning investigation.

See [docs/TRANSFER_LEARNING_INVESTIGATION.md](../docs/TRANSFER_LEARNING_INVESTIGATION.md) for full context and plan.

## Setup

In [ ]:
import os
if not os.path.exists('RETINNA'):
    !git clone https://github.com/scerruti/RETINNA.git

import sys
sys.path.insert(0, '/content/RETINNA' if '/content' in os.getcwd() else 'RETINNA')

In [ ]:
%pip install -q torchgeo h5py numpy tqdm

In [ ]:
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm

from src.colab_utils import setup_cabuaur_cached

## Load CaBuAr Dataset

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted")

# Setup CaBuAr cache
root = setup_cabuaur_cached()
print(f"✓ CaBuAr cache: {root}")

In [ ]:
from torchgeo.datasets import CaBuAr

# Load full dataset (all splits)
print("Loading CaBuAr dataset...")
cabuaur = CaBuAr(root=root, download=True, split='all')
print(f"✓ Loaded {len(cabuaur)} total samples")

# Get sample to verify structure
sample = cabuaur[0]
print(f"\nSample structure:")
print(f"  image shape: {sample['image'].shape}")
print(f"  mask shape: {sample['mask'].shape}")
print(f"  image dtype: {sample['image'].dtype}")
print(f"  image range: [{sample['image'].min():.1f}, {sample['image'].max():.1f}]")
print(f"\nBands in image [2, 12, 512, 512]:")
print(f"  Timestep 0 (pre-fire): 12 bands")
print(f"  Timestep 1 (post-fire): 12 bands")
print(f"\nTarget bands for RGB-IR:")
print(f"  B02 (Blue) - index 1")
print(f"  B03 (Green) - index 2")
print(f"  B04 (Red) - index 3")
print(f"  B08 (NIR) - index 7")

## Extract RGB-IR Bands

In [ ]:
def extract_rgbir_bands(image):
    """
    Extract RGB-IR (4 bands) from Sentinel-2 image.
    
    Args:
        image: [2, 12, 512, 512] (timesteps, bands, height, width)
        
    Returns:
        rgbir: [2, 4, 512, 512] (timesteps, RGB-IR bands, height, width)
        
    Band indices:
        1: B02 (Blue)
        2: B03 (Green)
        3: B04 (Red)
        7: B08 (NIR)
    """
    band_indices = [1, 2, 3, 7]  # Blue, Green, Red, NIR
    rgbir = image[:, band_indices, :, :]
    return rgbir

# Test extraction
test_rgbir = extract_rgbir_bands(sample['image'])
print(f"✓ RGB-IR extraction test:")
print(f"  Original shape: {sample['image'].shape}")
print(f"  RGB-IR shape: {test_rgbir.shape}")
print(f"  RGB-IR range: [{test_rgbir.min():.1f}, {test_rgbir.max():.1f}]")
print(f"  Expected shape: [2, 4, 512, 512] ✓" if test_rgbir.shape == (2, 4, 512, 512) else "✗ Shape mismatch!")

## Create RGB-IR HDF5 Dataset

In [ ]:
# Output path
output_dir = Path('/content/drive/MyDrive/RETINNA_DATA/cabuaur_rgbir')
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / 'cabuaur_rgbir.hdf5'

print(f"Creating RGB-IR dataset at: {output_file}")
print(f"Total samples: {len(cabuaur)}\n")

# Load splits from original dataset
cabuaur_train = CaBuAr(root=root, split='train')
cabuaur_val = CaBuAr(root=root, split='val')
cabuaur_test = CaBuAr(root=root, split='test')

train_size = len(cabuaur_train)
val_size = len(cabuaur_val)
test_size = len(cabuaur_test)
total_size = train_size + val_size + test_size

print(f"Split sizes:")
print(f"  Train: {train_size}")
print(f"  Val: {val_size}")
print(f"  Test: {test_size}")
print(f"  Total: {total_size}\n")

# Create HDF5 structure
with h5py.File(output_file, 'w') as f:
    # Create datasets
    images_ds = f.create_dataset('images', shape=(total_size, 2, 4, 512, 512), dtype=np.float32,
                                  chunks=(1, 2, 4, 512, 512), compression='gzip')
    masks_ds = f.create_dataset('masks', shape=(total_size, 1, 512, 512), dtype=np.uint8,
                                 chunks=(1, 1, 512, 512), compression='gzip')
    splits_ds = f.create_dataset('splits', shape=(total_size,), dtype=h5py.string_dtype(encoding='utf-8'))
    
    print("Extracting RGB-IR bands...\n")
    
    idx = 0
    
    # Process train split
    for i in tqdm(range(train_size), desc='Train', unit='samples'):
        sample = cabuaur_train[i]
        rgbir = extract_rgbir_bands(sample['image'])
        mask = sample['mask']
        
        images_ds[idx] = rgbir
        masks_ds[idx] = mask
        splits_ds[idx] = 'train'
        idx += 1
    
    # Process val split
    for i in tqdm(range(val_size), desc='Val', unit='samples'):
        sample = cabuaur_val[i]
        rgbir = extract_rgbir_bands(sample['image'])
        mask = sample['mask']
        
        images_ds[idx] = rgbir
        masks_ds[idx] = mask
        splits_ds[idx] = 'val'
        idx += 1
    
    # Process test split
    for i in tqdm(range(test_size), desc='Test', unit='samples'):
        sample = cabuaur_test[i]
        rgbir = extract_rgbir_bands(sample['image'])
        mask = sample['mask']
        
        images_ds[idx] = rgbir
        masks_ds[idx] = mask
        splits_ds[idx] = 'test'
        idx += 1
    
    # Add metadata
    f.attrs['description'] = 'RGB-IR (4-channel) extracted from CaBuAr Sentinel-2'
    f.attrs['bands'] = 'B02 (Blue), B03 (Green), B04 (Red), B08 (NIR)'
    f.attrs['shape'] = 'samples, timesteps, channels, height, width'
    f.attrs['train_count'] = train_size
    f.attrs['val_count'] = val_size
    f.attrs['test_count'] = test_size
    f.attrs['total_count'] = total_size

print(f"\n✓ RGB-IR dataset created successfully!")
print(f"  File: {output_file}")
print(f"  Size: {output_file.stat().st_size / (1024**3):.2f} GB")

## Validate Extracted Data

In [ ]:
# Validate by loading a few samples
print("Validating extracted data...\n")

with h5py.File(output_file, 'r') as f:
    print(f"Dataset info:")
    print(f"  Images shape: {f['images'].shape}")
    print(f"  Masks shape: {f['masks'].shape}")
    print(f"  Splits shape: {f['splits'].shape}")
    print(f"\nMetadata:")
    for key, val in f.attrs.items():
        print(f"  {key}: {val}")
    
    # Sample validation
    print(f"\nSample validation:")
    for idx in [0, len(f['images'])//2, len(f['images'])-1]:
        img = f['images'][idx]
        msk = f['masks'][idx]
        spl = f['splits'][idx]
        print(f"  Sample {idx} ({spl}): image {img.shape}, mask {msk.shape}, "
              f"img range [{img.min():.3f}, {img.max():.3f}], "
              f"mask sum {msk.sum()} pixels")

print(f"\n✓ Validation complete!")
print(f"✓ RGB-IR dataset ready for training")
print(f"\nNext steps:")
print(f"  1. Create RGB-IR DataLoader (optional)")
print(f"  2. Train U-Net on RGB-IR (modify 03_training.ipynb)")
print(f"  3. Compare performance vs 24-channel baseline")
print(f"  4. Prepare for NAIP transfer learning")